# AI Engineer Assessment Chatbot

## 1. Setup and Dependencies

This project uses:
- Groq for hosted LLM inference
- Superhero API for superhero information
- TF-IDF for text retrieval
- FastAPI for the `/ask` endpoint

### API Configuration

Create a `.env` file in the project root:

GROQ_API_KEY=your_key

SUPERHERO_API_TOKEN=your_token



## 1.1 Install dependencies


In [1]:
!pip install -q -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1.2 Imports

In [2]:
import json
import requests
from enum import Enum
from typing import Optional

from pydantic import BaseModel, Field
from groq import Groq

## 2. API Keys and Configuration

### 2.1 API Keys


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
SUPERHERO_API_TOKEN = os.getenv("SUPERHERO_API_TOKEN")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set.")

if not SUPERHERO_API_TOKEN:
    raise ValueError("SUPERHERO_API_TOKEN is not set.")

print("Groq API key loaded:", GROQ_API_KEY is not None)
print("Superhero API token loaded:", SUPERHERO_API_TOKEN is not None)

Groq API key loaded: True
Superhero API token loaded: True


In [4]:
client = Groq(api_key=GROQ_API_KEY)

## 3. Information Sources


### 3.1 Superhero API

In [5]:

SUPERHERO_API_BASE_URL = "https://superheroapi.com/api"


def search_superhero(name: str) -> dict:
    """
    Search for a superhero using the Superhero API.
    """

    url = f"{SUPERHERO_API_BASE_URL}/{SUPERHERO_API_TOKEN}/search/{name}"

    try:
        response = requests.get(
            url,
            timeout=10
        )

        response.raise_for_status()

        return response.json()

    except requests.RequestException as e:
        raise RuntimeError(
            f"Superhero API request failed: {str(e)}"
        )

### Test the Superhero API

In [6]:
result = search_superhero("Batman")

print("API response:", result.get("response"))
print("Number of results:", len(result.get("results", [])))

RuntimeError: Superhero API request failed: HTTPSConnectionPool(host='www.superheroapi.com', port=443): Max retries exceeded with url: /api.php/8dbfa65b621a1ca01d86e77daa063b2f/search/Batman (Caused by ConnectTimeoutError(<HTTPSConnection(host='www.superheroapi.com', port=443) at 0x2b590cc04d0>, 'Connection to www.superheroapi.com timed out. (connect timeout=10)'))

### 3.2 Text Dataset

In [ ]:
text_documents = {
    "rag.txt": """
Retrieval-Augmented Generation (RAG) is a technique that combines
information retrieval with text generation. Instead of relying only
on the knowledge stored in a language model, RAG retrieves relevant
documents from an external knowledge source and provides them to the
language model as context.

A typical RAG system has two main stages. First, a retriever searches
a collection of documents and finds the most relevant passages.
Second, a language model uses the retrieved passages to generate
an answer.

RAG is useful when the information comes from a private, changing,
or domain-specific knowledge base.
""",

    "llm.txt": """
Large Language Models (LLMs) are machine learning models trained on
large collections of text. They can perform tasks such as text
generation, summarization, question answering, and information
extraction.

LLMs generate responses based on patterns learned during training.
They can also be provided with external context at inference time,
for example through Retrieval-Augmented Generation.
""",

    "machine_learning.txt": """
Machine learning is a branch of artificial intelligence in which
models learn patterns from data.

Supervised learning uses labeled examples to train a model.
Unsupervised learning works with data without explicit labels.
Common supervised learning tasks include classification and
regression.

A machine learning pipeline commonly includes data preparation,
feature processing, model training, validation, and evaluation.
"""
}

print("Number of documents:", len(text_documents))

for name in text_documents:
    print("-", name)

Number of documents: 3
- rag.txt
- llm.txt
- machine_learning.txt


## 4. Text Processing and Retrieval


### 4.1 Chunking

In [ ]:
def chunk_text(text: str, chunk_size: int = 500):
    """
    Split text into smaller chunks.

    Each chunk will later be searchable by the retriever.
    """
    
    words = text.split()
    
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks


text_chunks = []

for document_name, document_text in text_documents.items():
    
    chunks = chunk_text(document_text)
    
    for chunk in chunks:
        text_chunks.append({
            "document": document_name,
            "content": chunk
        })


print("Number of chunks:", len(text_chunks))

for chunk in text_chunks:
    print("\nDocument:", chunk["document"])
    print(chunk["content"][:200], "...")

Number of chunks: 3

Document: rag.txt
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying only on the knowledge stored in a language model, RAG retrieves relevan ...

Document: llm.txt
Large Language Models (LLMs) are machine learning models trained on large collections of text. They can perform tasks such as text generation, summarization, question answering, and information extrac ...

Document: machine_learning.txt
Machine learning is a branch of artificial intelligence in which models learn patterns from data. Supervised learning uses labeled examples to train a model. Unsupervised learning works with data with ...


### 4.2 TF-IDF index

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
chunk_contents = [
    chunk["content"]
    for chunk in text_chunks
]

vectorizer = TfidfVectorizer(
    stop_words="english"
)

text_matrix = vectorizer.fit_transform(chunk_contents)

print("Number of chunks:", len(chunk_contents))
print("TF-IDF matrix shape:", text_matrix.shape)

Number of chunks: 3
TF-IDF matrix shape: (3, 88)


### 4.3 Text Retriever

In [ ]:
def retrieve_text(
    question: str,
    top_k: int = 3,
    threshold: float = 0.1
):
    """
    Retrieve the most relevant text chunks for a question.
    """

    # Convert the question into the same TF-IDF space
    query_vector = vectorizer.transform([question])

    # Compare the question with every text chunk
    similarity_scores = cosine_similarity(
        query_vector,
        text_matrix
    )[0]

    # Rank chunks from most relevant to least relevant
    ranked_indices = similarity_scores.argsort()[::-1]

    results = []

    for index in ranked_indices[:top_k]:

        score = float(similarity_scores[index])

        # Only keep sufficiently relevant chunks
        if score >= threshold:

            results.append({
                "document": text_chunks[index]["document"],
                "content": text_chunks[index]["content"],
                "score": score
            })

    return results

### Testing retriever

In [ ]:
test_questions = [
    "What is retrieval augmented generation?",
    "What are large language models?",
    "What is supervised learning?",
    "Who is Batman?"
]

for question in test_questions:

    results = retrieve_text(question)

    print("\nQuestion:", question)

    if not results:
        print("No relevant text found.")
        continue

    for result in results:
        print(
            f"Document: {result['document']} | "
            f"Score: {result['score']:.3f}"
        )


Question: What is retrieval augmented generation?
Document: llm.txt | Score: 0.293
Document: rag.txt | Score: 0.241

Question: What are large language models?
Document: llm.txt | Score: 0.425
Document: rag.txt | Score: 0.130

Question: What is supervised learning?
Document: machine_learning.txt | Score: 0.511

Question: Who is Batman?
No relevant text found.


##  5. Routing

### 5.1  Route definition

In [ ]:
class Route(str, Enum):
    TEXT = "text"
    SUPERHERO = "superhero"
    BOTH = "both"
    UNKNOWN = "unknown"

##  5.2 Routing Decision

In [ ]:
class RoutingDecision(BaseModel):
    route: Route
    superhero_name: Optional[str] = None
    reason: str = Field(min_length=1, max_length=300)

### 5.3 LLM-Based Router

In [ ]:
ROUTER_SYSTEM_PROMPT = """
You are a routing component for a question-answering system.

The system has two information sources:

1. TEXT:
   A local text dataset containing domain-specific information.

2. SUPERHERO:
   The Superhero API, which contains information about superhero characters.

Your job is to determine which source or sources are needed to answer the user's question.

Choose exactly one route:

- "text": The question can be answered using the text dataset.
- "superhero": The question requires superhero information.
- "both": The question requires information from both sources.
- "unknown": Neither source is sufficient or the intent is unclear.

If the question refers to a superhero, identify the superhero name when possible.

Return ONLY valid JSON with this structure:

{
    "route": "text | superhero | both | unknown",
    "superhero_name": "string or null",
    "reason": "short explanation"
}
"""


def llm_route(question: str) -> RoutingDecision:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": ROUTER_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": question
            }
        ],
        response_format={"type": "json_object"}
    )

    raw_output = response.choices[0].message.content

    decision = RoutingDecision.model_validate(
        json.loads(raw_output)
    )

    return decision

## 6. Deterministic Route Verification

### 6.1 verify_route()

In [ ]:
def verify_route(
    question: str,
    decision: RoutingDecision
) -> RoutingDecision:

    # Verify TEXT source
    if decision.route == Route.TEXT:

        text_results = retrieve_text(question)

        if not text_results:
            return RoutingDecision(
                route=Route.UNKNOWN,
                superhero_name=None,
                reason="The text route was proposed, but no relevant text was found."
            )

        return RoutingDecision(
            route=Route.TEXT,
            superhero_name=None,
            reason="The proposed text route was verified by the text retriever."
        )

    # Verify SUPERHERO source
    if decision.route == Route.SUPERHERO:

        if not decision.superhero_name:
            return RoutingDecision(
                route=Route.UNKNOWN,
                superhero_name=None,
                reason="The superhero route was proposed, but no superhero name was identified."
            )

        try:
            superhero_data = search_superhero(
                decision.superhero_name
            )

            if superhero_data.get("response") != "success":
                return RoutingDecision(
                    route=Route.UNKNOWN,
                    superhero_name=decision.superhero_name,
                    reason="The superhero could not be verified using the Superhero API."
                )

            return RoutingDecision(
                route=Route.SUPERHERO,
                superhero_name=decision.superhero_name,
                reason="The proposed superhero route was verified using the Superhero API."
            )

        except RuntimeError:
            return RoutingDecision(
                route=Route.UNKNOWN,
                superhero_name=decision.superhero_name,
                reason="The Superhero API could not be reached."
            )

    # Verify BOTH sources
    if decision.route == Route.BOTH:

        if not decision.superhero_name:
            return RoutingDecision(
                route=Route.UNKNOWN,
                superhero_name=None,
                reason="The both route was proposed, but no superhero name was identified."
            )

        # Verify text source
        text_results = retrieve_text(question)

        if not text_results:
            return RoutingDecision(
                route=Route.UNKNOWN,
                superhero_name=decision.superhero_name,
                reason="The both route was proposed, but no relevant text was found."
            )

        # Verify superhero source
        try:
            superhero_data = search_superhero(
                decision.superhero_name
            )

            if superhero_data.get("response") != "success":
                return RoutingDecision(
                    route=Route.UNKNOWN,
                    superhero_name=decision.superhero_name,
                    reason="The both route was proposed, but the superhero could not be verified."
                )

        except RuntimeError:
            return RoutingDecision(
                route=Route.UNKNOWN,
                superhero_name=decision.superhero_name,
                reason="The both route was proposed, but the Superhero API could not be reached."
            )

        return RoutingDecision(
            route=Route.BOTH,
            superhero_name=decision.superhero_name,
            reason="Both the text source and Superhero API verified the proposed route."
        )

    # UNKNOWN remains UNKNOWN
    return decision

### 6.2 hybrid_route()

In [ ]:
def hybrid_route(question: str) -> RoutingDecision:

    # Stage 1: LLM proposes the route
    llm_decision = llm_route(question)

    # Stage 2: deterministic verification
    verified_decision = verify_route(
        question,
        llm_decision
    )

    return verified_decision

### 6.3 Test Hybrid Routing

In [ ]:
routing_test_questions = [
    "What is retrieval augmented generation?",
    "Who is Batman?",
    "What is RAG and could Batman use it?",
    "What is the capital of France?"
]

for question in routing_test_questions:
    decision = hybrid_route(question)

    print("\nQuestion:", question)
    print("Route:", decision.route.value)
    print("Superhero:", decision.superhero_name)
    print("Reason:", decision.reason)


Question: What is retrieval augmented generation?
Route: text
Superhero: None
Reason: The proposed text route was verified by the text retriever.

Question: Who is Batman?
Route: unknown
Superhero: Batman
Reason: The Superhero API could not be reached.

Question: What is RAG and could Batman use it?
Route: unknown
Superhero: Batman
Reason: The both route was proposed, but the Superhero API could not be reached.

Question: What is the capital of France?
Route: unknown
Superhero: None
Reason: The question asks for general knowledge not covered by the superhero API or the domain-specific text dataset.


## 7. Source context

### 7.1 Superhero  context

In [ ]:
def get_superhero_context(superhero_name: str) -> dict:
    """
    Retrieve superhero information from the Superhero API.
    """

    data = search_superhero(superhero_name)

    if data.get("response") != "success":
        raise RuntimeError(
            f"Superhero '{superhero_name}' was not found."
        )

    results = data.get("results", [])

    if not results:
        raise RuntimeError(
            f"No results found for superhero '{superhero_name}'."
        )

    # Prefer an exact name match when available
    # If multiple exact matches exist, use the first API result
    # as the deterministic selection policy.
    exact_matches = [
        result
        for result in results
        if result.get("name", "").lower() == superhero_name.lower()
    ]

    superhero = (
        exact_matches[0]
        if exact_matches
        else results[0]
    )

    return superhero

In [ ]:
superhero = get_superhero_context("Batman")

print("Name:", superhero["name"])
print("Full name:", superhero["biography"]["full-name"])
print("Publisher:", superhero["biography"]["publisher"])
print("Powerstats:", superhero["powerstats"])

RuntimeError: Superhero API request failed: HTTPSConnectionPool(host='www.superheroapi.com', port=443): Max retries exceeded with url: /api.php/8dbfa65b621a1ca01d86e77daa063b2f/search/Batman (Caused by ConnectTimeoutError(<HTTPSConnection(host='www.superheroapi.com', port=443) at 0x193e3b8f310>, 'Connection to www.superheroapi.com timed out. (connect timeout=10)'))

### 7.2 Text Context

In [ ]:
def get_text_context(question: str) -> list:
    """
    Retrieve relevant text chunks for the question.
    """

    results = retrieve_text(
        question=question,
        top_k=3,
        threshold=0.1
    )

    return results

In [ ]:
text_context = get_text_context(
    "What is retrieval augmented generation?"
)

for item in text_context:

    print("Source:", item["document"])
    print("Score:", round(item["score"], 3))
    print("Content:", item["content"][:300])
    print()

Source: llm.txt
Score: 0.293
Content: Large Language Models (LLMs) are machine learning models trained on large collections of text. They can perform tasks such as text generation, summarization, question answering, and information extraction. LLMs generate responses based on patterns learned during training. They can also be provided w

Source: rag.txt
Score: 0.241
Content: Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying only on the knowledge stored in a language model, RAG retrieves relevant documents from an external knowledge source and provides them to the language model as context. A 



## 8. Answer Generation

 ### 8.1 Prompt

In [ ]:
ANSWER_SYSTEM_PROMPT = """
You are a helpful question-answering assistant.

Answer the user's question using ONLY the information provided
in the source context.

Do not invent facts that are not present in the source context.

If the provided sources do not contain enough information to answer
the question, clearly say that the available sources are insufficient.

Every answer MUST include a "Sources" section that identifies where
the information came from.

For text sources, use the document filename.

For superhero information, identify the source as:
Superhero API.
"""

### 8.2 Generate an Answer

In [ ]:
def generate_answer(
    question: str,
    source_context: str
) -> str:
    """
    Generate the final answer using the hosted LLM.
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": ANSWER_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"""
Question:
{question}

Source context:
{source_context}
"""
            }
        ]
    )

    return response.choices[0].message.content

### 8.3 Test the Answer Generator

In [ ]:
test_context = """
Source: rag.txt

Retrieval-Augmented Generation (RAG) is a technique that combines
information retrieval with text generation. A RAG system retrieves
relevant documents from an external knowledge source and provides
them to the language model as context.
"""

answer = generate_answer(
    question="What is retrieval augmented generation?",
    source_context=test_context
)

print(answer)

Retrieval Augmented Generation (RAG) is a technique that combines information retrieval with text generation. A RAG system retrieves relevant documents from an external knowledge source and provides them to the language model as context.  

**Sources**  
- rag.txt



## 9. End-to-End Pipeline

### 9.1  Answer Question

In [ ]:
def answer_question(question: str) -> dict:
    """
    Process a user question through the complete chatbot pipeline.

    Returns the generated answer and the sources used.
    """

    # Step 1: Determine and verify the route
    decision = hybrid_route(question)

    # If the route cannot be verified
    if decision.route == Route.UNKNOWN:
        return {
            "answer": (
                "I could not determine which available source "
                "can reliably answer this question."
            ),
            "sources": []
        }

    source_parts = []
    sources = []

    # Step 2: Retrieve text information
    if decision.route in {Route.TEXT, Route.BOTH}:

        text_results = get_text_context(question)

        for result in text_results:

            source_parts.append(
                f"Source: {result['document']}\n"
                f"Content: {result['content']}"
            )

            if result["document"] not in sources:
                sources.append(result["document"])

    # Step 3: Retrieve superhero information
    if decision.route in {Route.SUPERHERO, Route.BOTH}:

        superhero = get_superhero_context(
            decision.superhero_name
        )

        source_parts.append(
            "Source: Superhero API\n"
            f"Name: {superhero.get('name')}\n"
            f"Powerstats: {superhero.get('powerstats')}\n"
            f"Biography: {superhero.get('biography')}\n"
            f"Appearance: {superhero.get('appearance')}\n"
            f"Work: {superhero.get('work')}\n"
            f"Connections: {superhero.get('connections')}"
        )

        sources.append("Superhero API")

    # Step 4: Combine retrieved information
    source_context = "\n\n".join(source_parts)

    # Step 5: Generate the final answer
    answer = generate_answer(
        question=question,
        source_context=source_context
    )

    return {
        "answer": answer,
        "sources": sources
    }

### 9.2 End-to-End Tests

In [ ]:
test_questions = [
    "What is retrieval augmented generation?",
    "Who is Batman?",
    "What powers does Superman have?",
    "What is RAG and could Batman use it?"
]

for question in test_questions:

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("=" * 80)

    answer = answer_question(question)

    print(answer)


QUESTION: What is retrieval augmented generation?
{'answer': 'Retrieval Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying solely on the knowledge stored in a language model, RAG retrieves relevant documents from an external knowledge source and supplies them to the language model as context. A typical RAG system has two main stages: first, a retriever searches a collection of documents and finds the most relevant passages; second, a language model uses those retrieved passages to generate an answer. RAG is especially useful when the needed information comes from a private, changing, or domain‑specific knowledge base.  \n\n**Sources**  \n- llm.txt  \n- rag.txt', 'sources': ['llm.txt', 'rag.txt']}

QUESTION: Who is Batman?
{'answer': 'Batman is a superhero from DC Comics.  \n- **Full name:** Terry\u202fMcGinnis  \n- **Aliases:** Batman\u202fII, The Tomorrow Knight, The second Dark Knight, The Dark Knight of Tomorrow, B

## 10. FastAPI Application

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

### 10.1  Request model

In [ ]:

class AskRequest(BaseModel):
    question: str = Field(
        description="Natural language question for the chatbot"
    )

### 10.2 Response model

In [ ]:
class AskResponse(BaseModel):
    question: str
    answer: str
    sources: list[str]

### 10.3 the FastAPI application

In [ ]:
app = FastAPI(
    title="AI Engineer Assessment Chatbot",
    description="Hybrid chatbot using a text dataset, Superhero API, and hosted LLM.",
    version="1.0.0"
)

### 10.4 Create / ask endpoint

In [ ]:
@app.post("/ask", response_model=AskResponse)
def ask(request: AskRequest):

    question = request.question.strip()

    # Validate empty questions
    if not question:
        raise HTTPException(
            status_code=400,
            detail="Question cannot be empty."
        )

    # Validate question length
    if len(question) > 1000:
        raise HTTPException(
            status_code=400,
            detail="Question is too long. Maximum length is 1000 characters."
        )

    try:
        result = answer_question(question)

        return AskResponse(
            question=question,
            answer=result["answer"],
            sources=result["sources"]
        )

    except RuntimeError as e:
        raise HTTPException(
            status_code=502,
            detail=str(e)
        )

    except Exception:
        raise HTTPException(
            status_code=500,
            detail="An unexpected error occurred while processing the question."
        )

### 10.5 test the endpoint

### Rag test

In [ ]:
from fastapi.testclient import TestClient

test_client = TestClient(app)

response = test_client.post(
    "/ask",
    json={
        "question": "What is retrieval augmented generation?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

c:\Users\GCB\Downloads\engineer_test\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Status code: 200
Response:
{'question': 'What is retrieval augmented generation?', 'answer': 'Retrieval Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying solely on the knowledge stored in a language model, RAG retrieves relevant documents from an external knowledge source and supplies them to the language model as context. A typical RAG system has two main stages: first, a retriever searches a collection of documents and finds the most relevant passages; second, a language model uses those retrieved passages to generate an answer. RAG is especially useful when the needed information comes from a private, changing, or domain‑specific knowledge base.  \n\n**Sources**  \n- llm.txt  \n- rag.txt', 'sources': ['llm.txt', 'rag.txt']}


### Batman test

In [ ]:
response = test_client.post(
    "/ask",
    json={
        "question": "Who is Batman?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'question': 'Who is Batman?', 'answer': 'Batman is a superhero from DC Comics.  \n- **Full name:** Terry\u202fMcGinnis  \n- **Aliases:** Batman\u202fII, The Tomorrow Knight, The second Dark Knight, The Dark Knight of Tomorrow, Batman\u202fBeyond  \n- **Place of birth:** Gotham City, 25th\u202fCentury  \n- **First appearance:** *Batman\u202fBeyond*\u202f#1  \n- **Alignment:** Good  \n- **Occupation:** (not specified)  \n- **Base:** 21st\u202fCentury Gotham City  \n- **Family:** Bruce\u202fWayne (biological father), Warren\u202fMcGinnis (father, deceased), Mary\u202fMcGinnis (mother), Matt\u202fMcGinnis (brother)  \n- **Affiliations:** Batman Family, Justice League Unlimited  \n\n**Powerstats**  \n- Intelligence\u202f81  \n- Strength\u202f40  \n- Speed\u202f29  \n- Durability\u202f55  \n- Power\u202f63  \n- Combat\u202f90  \n\n**Appearance**  \n- Gender: Male  \n- Race: Human  \n- Height: 5\'10" (178\u202fcm)  \n- Weight: 170\u202flb (77\u202fkg)  \n- Eye colo

### Both test 

In [ ]:
response = test_client.post(
    "/ask",
    json={
        "question": "What is RAG and could Batman use it?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'question': 'What is RAG and could Batman use it?', 'answer': '**What is RAG?**  \nRetrieval‑Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying only on the knowledge stored in a language model, RAG retrieves relevant documents from an external knowledge source and provides them to the language model as context. A typical RAG system has two main stages:  \n1. **Retriever** – searches a collection of documents and finds the most relevant passages.  \n2. **Generator** – uses the retrieved passages to generate an answer.  \nRAG is especially useful when the information comes from a private, changing, or domain‑specific knowledge base.  \n\n**Could Batman use RAG?**  \nThe provided sources do not contain any explicit information about Batman’s ability or willingness to use a Retrieval‑Augmented Generation system. While Batman’s intelligence score is 81, the sources do not state whether he has acc

## 11. Error Handling Tests

The chatbot should handle invalid input and external service failures gracefully.

We test:
- Empty questions
- Questions longer than 1000 characters
- Superhero API failure
- Hosted LLM failure
- Unexpected application failure


### Test empty question

In [ ]:

response = test_client.post(
    "/ask",
    json={
        "question": ""
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 400
Response:
{'detail': 'Question cannot be empty.'}


### Test a question longer than 1000 characters


In [ ]:

long_question = "What is RAG? " * 100

response = test_client.post(
    "/ask",
    json={
        "question": long_question
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 400
Response:
{'detail': 'Question is too long. Maximum length is 1000 characters.'}


### Temporarily simulate a Superhero API failure


In [ ]:

original_search_superhero = search_superhero

def failing_search_superhero(name: str) -> dict:
    raise RuntimeError("Simulated Superhero API failure")


search_superhero = failing_search_superhero

### Test how /ask handles the Superhero API failure


In [ ]:

response = test_client.post(
    "/ask",
    json={
        "question": "Who is Batman?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'question': 'Who is Batman?', 'answer': 'I could not determine which available source can reliably answer this question.', 'sources': []}


### Restore the original Superhero API function


In [ ]:
search_superhero = original_search_superhero

### Test hosted LLM failure

In [ ]:
# Save the original LLM function
original_generate_answer = generate_answer

In [ ]:
# Temporarily simulate a hosted LLM failure
def failing_generate_answer(
    question: str,
    source_context: str
) -> str:
    raise RuntimeError("Simulated LLM API failure")

In [ ]:
generate_answer = failing_generate_answer

### Test how `/ask` handles a hosted LLM failure.

In [ ]:
response = test_client.post(
    "/ask",
    json={"question": "What is RAG?"}
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 502
Response:
{'detail': 'Simulated LLM API failure'}


### Restore the original LLM function

In [ ]:
generate_answer = original_generate_answer

### Test unexpected application failure

In [ ]:
original_answer_question = answer_question

def failing_answer_question(question: str) -> dict:
    raise ValueError("Simulated unexpected failure")

answer_question = failing_answer_question

### Test how `/ask` handles an unexpected application failure

In [ ]:
response = test_client.post(
    "/ask",
    json={
        "question": "What is RAG?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 500
Response:
{'detail': 'An unexpected error occurred while processing the question.'}


In [ ]:
answer_question = original_answer_question

### Test normal request after restoring all services

In [ ]:
response = test_client.post(
    "/ask",
    json={
        "question": "What is retrieval augmented generation?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'question': 'What is retrieval augmented generation?', 'answer': 'Retrieval Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying solely on the knowledge stored in a language model, RAG retrieves relevant documents from an external knowledge source and supplies them to the language model as context. A typical RAG system has two main stages: first, a retriever searches a collection of documents and finds the most relevant passages; second, a language model uses those retrieved passages to generate an answer. RAG is especially useful when the needed information comes from a private, changing, or domain‑specific knowledge base.  \n\n**Sources**  \n- llm.txt  \n- rag.txt', 'sources': ['llm.txt', 'rag.txt']}
